# **DATA PROCESSING - Hướng dẫn Chi tiết**

Notebook này thực hiện một quy trình xử lý dữ liệu đầy đủ từ **MovieLens Dataset** (bộ dữ liệu về cách người dùng đánh giá phim) bao gồm 6 bước chính:

1. **Review Dataset** - Tải và kiểm tra dữ liệu từ 5 file CSV
2. **Data Cleaning** - Làm sạch dữ liệu (xử lý thiếu, trùng, bất thường)
3. **Data Integration** - Gộp nhiều bảng thành một bảng tổng hợp
4. **Data Reduction** - Lựa chọn những cột quan trọng
5. **Data Transformation** - Biến đổi định dạng và chuẩn hóa dữ liệu
6. **Data Discretization** - Rời rạc hóa dữ liệu liên tục thành danh mục

# **DATA PROCESSING**

## **I.Review Dataset**

### **Nhập các thư viện cần thiết**

**Mục đích**: Chuẩn bị các công cụ để xử lý dữ liệu, gọi API và biến đổi dữ liệu.

- **pandas, numpy**: Xử lý và phân tích dữ liệu dạng bảng
- **requests**: Gọi API TMDb để lấy thông tin phim
- **time**: Tạo độ trễ giữa các request API (tránh bị rate limit)
- **MinMaxScaler**: Chuẩn hóa dữ liệu số về khoảng [0, 1]
- **re**: Sử dụng Regex để trích xuất thông tin (ví dụ: năm từ title)
- **os**: Tương tác với hệ thống file
- **TfidfVectorizer**: Dùng để chuyển dữ liệu văn bản thành dạng số
- **seaborn, matplotlib**: Vẽ các biểu đồ

In [2]:
#LIBRARY
import pandas as pd
import numpy as np
import requests
import time
from sklearn.preprocessing import MinMaxScaler
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer

### **Hàm kiểm tra dữ liệu (inspect_data) - Công cụ hỗ trợ**

**Chức năng**: Đây là một hàm hỗ trợ để kiểm tra toàn diện mỗi bộ dữ liệu CSV trước khi xử lý.

**Các thông tin được hiển thị**:
1. **Thông tin tổng quan** - Cấu trúc dữ liệu, kiểu dữ liệu của mỗi cột
2. **Dữ liệu thiếu (Null)** - Danh sách các cột có giá trị null
3. **Trùng lặp (Duplicates)** - Kiểm tra dòng trùng toàn bộ và trùng theo khóa ID
4. **Giá trị duy nhất** - Số lượng giá trị riêng biệt trong mỗi cột
5. **Thống kê** - Min, Max, Mean, Std Dev cho các cột số
6. **Mẫu dữ liệu** - 5 dòng đầu tiên để xem ngoại hình

**Lợi ích**: Giúp phát hiện sớm các vấn đề như thiếu dữ liệu, dữ liệu bất thường, hoặc lỗi trong quá trình tải dữ liệu.

In [3]:
def inspect_data(df, name="Dataset"):
    print(f"{'='*40}")
    print(f" KIỂM TRA: {name.upper()}")
    print(f"{'='*40}")
    
    # 1. Thông tin tổng quan
    print("\n1. THÔNG TIN TỔNG QUAN:")
    df.info()
    
    # 2. Dữ liệu thiếu (Null)
    print("\n2. DỮ LIỆU THIẾU (NULL):")
    null_count = df.isnull().sum()
    if null_count.sum() == 0:
        print("-> Không có dữ liệu thiếu.")
    else:
        print(null_count[null_count > 0])
        
    # 3. KIỂM TRA TRÙNG LẶP (DUPLICATES) - MỚI
    print("\n3. KIỂM TRA TRÙNG LẶP (DUPLICATES):")
    # Trùng toàn bộ dòng
    total_dupes = df.duplicated().sum()
    print(f" - Số dòng trùng lặp hoàn toàn: {total_dupes}")
    
    # Trùng theo ID (Nếu có các cột ID phổ biến)
    id_cols = [col for col in ['movieId', 'userId', 'tmdbId', 'imdbId'] if col in df.columns]
    for col in id_cols:
        dupe_ids = df.duplicated(subset=[col]).sum()
        if dupe_ids > 0:
            print(f" - Cảnh báo: Cột '{col}' bị trùng {dupe_ids} giá trị (có thể là lỗi nếu đây là khóa chính)")

    # 4. Giá trị duy nhất (Unique)
    print("\n4. GIÁ TRỊ DUY NHẤT (UNIQUE):")
    for col in df.columns:
        print(f" - Cột '{col}': {df[col].nunique()} giá trị duy nhất")
        
    # 5. Thống kê số liệu
    print("\n5. THỐNG KÊ SỐ LIỆU:")
    display(df.describe())
    
    # 6. Dữ liệu mẫu
    print("\n6. DỮ LIỆU MẪU (HEAD):")
    display(df.head())
    print("\n")

### **links.csv**

**Mô tả**: Đóng vai trò là "từ điển" để kết nối mã phim giữa các nền tảng khác nhau.

- movieId: ID của MovieLens.

- imdbId: ID của phim trên trang IMDb.

- tmdbId: ID của phim trên trang TMDb (dùng để gọi API).

In [4]:
df_links = pd.read_csv('links.csv')
inspect_data(df_links, "Links Dataset")

 KIỂM TRA: LINKS DATASET

1. THÔNG TIN TỔNG QUAN:
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB

2. DỮ LIỆU THIẾU (NULL):
tmdbId    8
dtype: int64

3. KIỂM TRA TRÙNG LẶP (DUPLICATES):
 - Số dòng trùng lặp hoàn toàn: 0
 - Cảnh báo: Cột 'tmdbId' bị trùng 8 giá trị (có thể là lỗi nếu đây là khóa chính)

4. GIÁ TRỊ DUY NHẤT (UNIQUE):
 - Cột 'movieId': 9742 giá trị duy nhất
 - Cột 'imdbId': 9742 giá trị duy nhất
 - Cột 'tmdbId': 9733 giá trị duy nhất

5. THỐNG KÊ SỐ LIỆU:


,movieId,imdbId,tmdbId
count,9742.000000,9.742000e+03,9734.000000
mean,42200.353623,6.771839e+05,55162.123793
std,52160.494854,1.107228e+06,93653.481487
min,1.000000,4.170000e+02,2.000000
25%,3248.250000,9.518075e+04,9665.500000
50%,7300.000000,1.672605e+05,16529.000000
75%,76232.000000,8.055685e+05,44205.750000
max,193609.000000,8.391976e+06,525662.000000



6. DỮ LIỆU MẪU (HEAD):


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


### **movies.csv**

**Mô tả**: Bảng tra cứu thông tin định danh cốt lõi cho mỗi bộ phim.

- movieId: Mã định danh phim (khớp với bảng ratings).

- title: Tên phim đầy đủ kèm năm phát hành.

- genres: Các thể loại phim (ngăn cách bởi dấu gạch đứng |).

In [5]:
df_movies = pd.read_csv('movies.csv')
inspect_data(df_movies, "Movies Dataset")

 KIỂM TRA: MOVIES DATASET

1. THÔNG TIN TỔNG QUAN:
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 228.5 KB

2. DỮ LIỆU THIẾU (NULL):
-> Không có dữ liệu thiếu.

3. KIỂM TRA TRÙNG LẶP (DUPLICATES):
 - Số dòng trùng lặp hoàn toàn: 0

4. GIÁ TRỊ DUY NHẤT (UNIQUE):
 - Cột 'movieId': 9742 giá trị duy nhất
 - Cột 'title': 9737 giá trị duy nhất
 - Cột 'genres': 951 giá trị duy nhất

5. THỐNG KÊ SỐ LIỆU:


,movieId
count,9742.000000
mean,42200.353623
std,52160.494854
min,1.000000
25%,3248.250000
50%,7300.000000
75%,76232.000000
max,193609.000000



6. DỮ LIỆU MẪU (HEAD):


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


### **ratings.csv**

**Mô tả**: Lưu trữ toàn bộ hành vi chấm điểm của người dùng, dùng để xác định sự tương đồng giữa các User.

- userId: Mã định danh duy nhất của người dùng.

- movieId: Mã định danh của phim (theo hệ thống MovieLens).

- rating: Điểm đánh giá (thường từ 0.5 đến 5.0).

- timestamp: Thời điểm người dùng thực hiện đánh giá (dạng Unix thời gian).

In [6]:
df_ratings = pd.read_csv('ratings.csv')
inspect_data(df_ratings, "Ratings Dataset")

 KIỂM TRA: RATINGS DATASET

1. THÔNG TIN TỔNG QUAN:
<class 'pandas.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB

2. DỮ LIỆU THIẾU (NULL):
-> Không có dữ liệu thiếu.

3. KIỂM TRA TRÙNG LẶP (DUPLICATES):
 - Số dòng trùng lặp hoàn toàn: 0
 - Cảnh báo: Cột 'movieId' bị trùng 91112 giá trị (có thể là lỗi nếu đây là khóa chính)
 - Cảnh báo: Cột 'userId' bị trùng 100226 giá trị (có thể là lỗi nếu đây là khóa chính)

4. GIÁ TRỊ DUY NHẤT (UNIQUE):
 - Cột 'userId': 610 giá trị duy nhất
 - Cột 'movieId': 9724 giá trị duy nhất
 - Cột 'rating': 10 giá trị duy nhất
 - Cột 'timestamp': 85043 giá trị duy nhất

5. THỐNG KÊ SỐ LIỆU:


,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09



6. DỮ LIỆU MẪU (HEAD):


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


### **tags.csv**

**Mô tả**: Chứa các từ khóa (tags) mà người dùng tự gán cho phim. Đây là dữ liệu cực kỳ quý giá để hiểu về "vibe" hoặc các đặc điểm ngách của phim mà thể loại (genres) không mô tả hết được (ví dụ: "cốt truyện hại não", "nhạc phim hay", "dark comedy").

- userId: Mã định danh của người dùng đã tạo tag đó.

- movieId: Mã định danh của phim được gắn tag.

- tag: Nội dung từ khóa hoặc cụm từ mô tả (ví dụ: "atmospheric", "superhero", "thought-provoking").

- timestamp: Thời điểm người dùng gắn thẻ cho phim.

In [229]:
df_tags = pd.read_csv('tags.csv')
inspect_data(df_tags, "Tags Dataset")

 KIỂM TRA: TAGS DATASET

1. THÔNG TIN TỔNG QUAN:
<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 115.2 KB

2. DỮ LIỆU THIẾU (NULL):
-> Không có dữ liệu thiếu.

3. KIỂM TRA TRÙNG LẶP (DUPLICATES):
 - Số dòng trùng lặp hoàn toàn: 0
 - Cảnh báo: Cột 'movieId' bị trùng 2111 giá trị (có thể là lỗi nếu đây là khóa chính)
 - Cảnh báo: Cột 'userId' bị trùng 3625 giá trị (có thể là lỗi nếu đây là khóa chính)

4. GIÁ TRỊ DUY NHẤT (UNIQUE):
 - Cột 'userId': 58 giá trị duy nhất
 - Cột 'movieId': 1572 giá trị duy nhất
 - Cột 'tag': 1589 giá trị duy nhất
 - Cột 'timestamp': 3411 giá trị duy nhất

5. THỐNG KÊ SỐ LIỆU:


,userId,movieId,timestamp
count,3683.000000,3683.000000,3.683000e+03
mean,431.149335,27252.013576,1.320032e+09
std,158.472553,43490.558803,1.721025e+08
min,2.000000,1.000000,1.137179e+09
25%,424.000000,1262.500000,1.137521e+09
50%,474.000000,4454.000000,1.269833e+09
75%,477.000000,39263.000000,1.498457e+09
max,610.000000,193565.000000,1.537099e+09



6. DỮ LIỆU MẪU (HEAD):


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


## **II. DATA CLEANING**

### **Handling Missing Values**

**links.csv**
- Sau khi xem qua thông tin thì cột "tmdbid" bị null nhưng nhờ cột "imdbId" có để gửi request api để lấy tmdbid

In [7]:
API_KEY = "4d3a1be95ec76ac97e468425e4f746a9"

def get_tmdb_id_from_imdb(imdb_id):
    """Hàm bổ trợ để gọi API TMDb dựa trên mã IMDb"""
    # Định dạng tt + 7 chữ số (ví dụ: 114709 -> tt0114709)
    formatted_imdb = f"tt{str(int(imdb_id)).zfill(7)}"
    url = f"https://api.themoviedb.org/3/find/{formatted_imdb}?api_key={API_KEY}&external_source=imdb_id"
    
    try:
        res = requests.get(url, timeout=5)
        if res.status_code == 200:
            results = res.json().get('movie_results', [])
            return results[0]['id'] if results else None
    except:
        return None
    return None

def fill_missing_tmdb_ids(df):
    """Hàm chính để cập nhật biến df_links"""
    # Tạo bản sao để tránh lỗi SettingWithCopyWarning
    df_result = df.copy()
    
    # Xác định các dòng có tmdbId là NaN
    mask = df_result['tmdbId'].isna()
    missing_indices = df_result[mask].index
    
    if len(missing_indices) == 0:
        print("Không có tmdbId nào bị thiếu!")
        return df_result

    print(f"Đang xử lý {len(missing_indices)} dòng thiếu tmdbId...")

    for idx in missing_indices:
        imdb_id = df_result.at[idx, 'imdbId']
        new_id = get_tmdb_id_from_imdb(imdb_id)
        
        if new_id:
            df_result.at[idx, 'tmdbId'] = new_id
            print(f" -> MovieId {df_result.at[idx, 'movieId']}: Đã điền tmdbId {int(new_id)}")
        else:
            print(f" -> MovieId {df_result.at[idx, 'movieId']}: Không tìm thấy trên TMDb")
            
        time.sleep(0.25) # Tránh bị giới hạn tốc độ API (TMDb: 40 req/10s)

    print("--- Hoàn tất cập nhật dữ liệu trên biến ---")
    return df_result

df_links = fill_missing_tmdb_ids(df_links)


Đang xử lý 8 dòng thiếu tmdbId...
 -> MovieId 791: Đã điền tmdbId 706007
 -> MovieId 1107: Đã điền tmdbId 1589361
 -> MovieId 2851: Không tìm thấy trên TMDb
 -> MovieId 4051: Không tìm thấy trên TMDb
 -> MovieId 26587: Đã điền tmdbId 37452
 -> MovieId 32600: Không tìm thấy trên TMDb
 -> MovieId 40697: Không tìm thấy trên TMDb
 -> MovieId 79299: Không tìm thấy trên TMDb
--- Hoàn tất cập nhật dữ liệu trên biến ---


### **Xử lý dữ liệu thiếu: Lấy ID từ API TMDb**

**Bối cảnh**: Cột `tmdbId` bị thiếu vài giá trị. Chúng ta cần nó để gọi API lấy thông tin metrics (budget, revenue, popularity...).

**Chiến lược**:
- Các phim thiếu `tmdbId` vẫn có `imdbId` (từ IMDb)
- Sử dụng IMDb ID để truy vấn TMDb API và lấy lại tmdbId

**Hai hàm chính**:
1. **`get_tmdb_id_from_imdb(imdb_id)`** 
   - Gọi API TMDb với IMDb ID
   - Trả về tmdbId nếu tìm thấy phim

2. **`fill_missing_tmdb_ids(df)`**
   - Lặp qua tất cả dòng thiếu tmdbId
   - Gọi hàm trên để điền giá trị
   - Thêm độ trễ 0.1 giây giữa mỗi request (tránh bị rate limit)

**Lưu ý**: Nếu không tìm thấy phim trên TMDb, sẽ thực hiện xửlý thủ công (xem các bước tiếp theo).

In [8]:
# Hiển thị các phim còn thiếu tmdbId sau khi gọi API
missing_movies = df_links[df_links['tmdbId'].isna()].merge(df_movies[['movieId', 'title']], on='movieId')
if len(missing_movies) > 0:

    print(f"Số phim vẫn còn thiếu tmdbId: {len(missing_movies)}")  

    print(missing_movies[['movieId', 'title', 'imdbId']])
    
else:
    print("Tất cả phim đã có tmdbId!")

Số phim vẫn còn thiếu tmdbId: 5
   movieId                                              title  imdbId
0     2851                                    Saturn 3 (1980)   81454
1     4051  Horrors of Spider Island (Ein Toter Hing im Ne...   56600
2    32600                                        Eros (2004)  377059
3    40697                                          Babylon 5  105946
4    79299         No. 1 Ladies' Detective Agency, The (2008)  874957


**Những phim đã tìm được**:

- Saturn 3(1980): https://www.themoviedb.org/movie/19761-saturn-3 với tmdbId là 19761

- Horrors of Spider Island: https://www.themoviedb.org/movie/31392-ein-toter-hing-im-netz với tmdbId là 31392

- Eros (2004): https://www.themoviedb.org/movie/39850-eros với tmdbId là 39850

- Babylon 5: https://www.themoviedb.org/movie/10941-babylon-5-thirdspace với tmdbId là 10941
 
- No. 1 Ladies' Detective Agency, The (2008): https://www.themoviedb.org/tv/7154-the-no-1-ladies-detective-agency với tmdbId là 7154

Cập nhật vào links.csv

In [9]:
df_links.loc[df_links['movieId'] == 2851, 'tmdbId'] = 19761
df_links.loc[df_links['movieId'] == 4051, 'tmdbId'] = 31392
df_links.loc[df_links['movieId'] == 32600, 'tmdbId'] = 39850
df_links.loc[df_links['movieId'] == 40697, 'tmdbId'] = 10941
df_links.loc[df_links['movieId'] == 79299, 'tmdbId'] = 7154

df_links['tmdbId'] = df_links['tmdbId'].astype(int) # Ép kiểu về số nguyên

In [10]:
inspect_data(df_links, "Movie Metrics Dataset")

 KIỂM TRA: MOVIE METRICS DATASET

1. THÔNG TIN TỔNG QUAN:
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   imdbId   9742 non-null   int64
 2   tmdbId   9742 non-null   int64
dtypes: int64(3)
memory usage: 228.5 KB

2. DỮ LIỆU THIẾU (NULL):
-> Không có dữ liệu thiếu.

3. KIỂM TRA TRÙNG LẶP (DUPLICATES):
 - Số dòng trùng lặp hoàn toàn: 0
 - Cảnh báo: Cột 'tmdbId' bị trùng 4 giá trị (có thể là lỗi nếu đây là khóa chính)

4. GIÁ TRỊ DUY NHẤT (UNIQUE):
 - Cột 'movieId': 9742 giá trị duy nhất
 - Cột 'imdbId': 9742 giá trị duy nhất
 - Cột 'tmdbId': 9738 giá trị duy nhất

5. THỐNG KÊ SỐ LIỆU:


,movieId,imdbId,tmdbId
count,9742.000000,9.742000e+03,9.742000e+03
mean,42200.353623,6.771839e+05,5.536748e+04
std,52160.494854,1.107228e+06,9.512902e+04
min,1.000000,4.170000e+02,2.000000e+00
25%,3248.250000,9.518075e+04,9.668000e+03
50%,7300.000000,1.672605e+05,1.654400e+04
75%,76232.000000,8.055685e+05,4.420575e+04
max,193609.000000,8.391976e+06,1.589361e+06



6. DỮ LIỆU MẪU (HEAD):


,movieId,imdbId,tmdbId
0,1,114709,862
1,2,113497,8844
2,3,113228,15602
3,4,114885,31357
4,5,113041,11862


### **Xử lý trùng lặp (Handling Duplicates)**

**Phân tích trùng lặp theo từng file**:

1. **Ratings & Tags**: Có thể chứa dữ liệu lặp (ví dụ: user 1 đánh giá movie A nhiều lần), nhưng điều này không hợp lệ. Cần xóa đánh giá cũ khi merge movieId.

2. **Links**: Mỗi movie chỉ nên có 1 dòng duy nhất. Nếu cột "tmdbId" bị trùng, điều này có thể chỉ ra dữ liệu trùng lặp. Cần kiểm tra và xử lý.

In [234]:
# Kiểm tra trùng lặp dựa trên tmdbId
duplicate_ids = df_links[df_links.duplicated(subset=['tmdbId'], keep=False)]
dupes_with_title = pd.merge(duplicate_ids, df_movies[['movieId', 'title']], on='movieId', how='left')


print("Các dòng có cùng tmdbId nhưng bị lặp:")
display(dupes_with_title.sort_values(by='tmdbId'))

Các dòng có cùng tmdbId nhưng bị lặp:


,movieId,imdbId,tmdbId,title
1,6003,290538,4912,Confessions of a Dangerous Mind (2002)
5,144606,270288,4912,Confessions of a Dangerous Mind (2002)
4,40697,105946,10941,Babylon 5
2,7812,121804,10941,Babylon 5: Thirdspace (1998)
7,168358,79285,19761,Saturn 3 (1980)
0,2851,81454,19761,Saturn 3 (1980)
3,32600,377059,39850,Eros (2004)
6,147002,343663,39850,Eros (2004)


In [235]:
ids_to_drop = [144606, 40697, 2851, 32600]
df_movies = df_movies[~df_movies['movieId'].isin(ids_to_drop)]
df_links = df_links[~df_links['movieId'].isin(ids_to_drop)]

In [236]:
# Cấu trúc: {ID_cũ: ID_mới}
mapping_dict = {
    144606: 6003,  # Ví dụ: Thay 144606 bằng 123
    40697: 7812,
    2851: 168358,
    32600: 147002
}

# Thay thế trong bảng Ratings
df_ratings['movieId'] = df_ratings['movieId'].replace(mapping_dict)

# Thay thế trong bảng Tags
df_tags['movieId'] = df_tags['movieId'].replace(mapping_dict)

print("Đã replace xong movieId ở các bảng tương tác.")

# Sắp xếp để giữ lại đánh giá mới nhất
df_ratings = df_ratings.sort_values(by=['userId', 'movieId', 'timestamp'], ascending=[True, True, False])

# Xóa trùng lặp (nếu user đánh giá cùng 1 ID phim sau khi đã replace)
df_ratings = df_ratings.drop_duplicates(subset=['userId', 'movieId'], keep='first')

print(f"Số lượng ratings sau khi merge ID và dọn dẹp: {len(df_ratings)}")


Đã replace xong movieId ở các bảng tương tác.
Số lượng ratings sau khi merge ID và dọn dẹp: 100834


### **Xử lý ID trùng lặp và đồng bộ hóa (Duplicate & Mapping Cleanup)**

**Vấn đề**: 
- Khi xóa các phim bị trùng (144606, 40697, 2851, 32600) khỏi bảng movies và links
- Nhưng các phim này vẫn có dữ liệu đánh giá (ratings) và tag trong các bảng khác
- Kết quả: Dữ liệu không đồng bộ (orphan data)

**Chiến lược xử lý**:

1. **Ánh xạ ID cũ → ID mới** (Mapping)
   - Thay thế các phim bị xóa bằng phim tương đương từ TMDb
   - Ví dụ: movieId 144606 → 6003
   - Áp dụng mapping này lên bảng ratings và tags

2. **Loại bỏ đánh giá lặp**
   - Nếu một user đánh giá cùng phim nhiều lần (sau khi merge ID)
   - Giữ lại đánh giá mới nhất (`sort by timestamp` rồi `drop_duplicates`)

3. **Kiểm tra tế nhị**
   - Thảo luận: Chỉ xóa khi chắc chắn trùng lặp hoàn toàn
   - Nếu phim c khác nhau, cảnh báo trước khi xóa

### **Loại bỏ dữ liệu nhiễu (Filtering Noise)**

Các giá trị trong dataset đều dương hợp lệ

Tiến hành kiểm tra timestamp hợp lệ: Cận dưới 0 (Ngày 01/01/1970 bắt đầu kỷ nguyên Unix) và Cận trên (now)

In [11]:
def validate_unix_timestamps(df, col='timestamp'):
    current_unix = int(time.time()) # Lấy thời gian hiện tại (năm 2026)
    
    # 1. Tìm các dòng có timestamp ở tương lai
    future_noise = df[df[col] > current_unix]
    
    # 2. Tìm các dòng có timestamp quá cũ (ví dụ trước năm 1990 - Unix < 631152000)
    # Tùy vào bộ dữ liệu, thường MovieLens bắt đầu từ 1995
    too_old_noise = df[df[col] < 0] 
    
    print(f"--- KIỂM TRA TIMESTAMP (UNIX) ---")
    print(f"Số lượng bản ghi ở tương lai: {len(future_noise)}")
    print(f"Số lượng bản ghi âm (trước 1970): {len(too_old_noise)}")
    
    return future_noise, too_old_noise

# Thực thi
print("ratings.csv")
print(validate_unix_timestamps(df_ratings))

print("tags.csv")
print(validate_unix_timestamps(df_tags))


ratings.csv
--- KIỂM TRA TIMESTAMP (UNIX) ---
Số lượng bản ghi ở tương lai: 0
Số lượng bản ghi âm (trước 1970): 0
(Empty DataFrame
Columns: [userId, movieId, rating, timestamp]
Index: [], Empty DataFrame
Columns: [userId, movieId, rating, timestamp]
Index: [])
tags.csv


NameError: name 'df_tags' is not defined

Trước khi bước tiếp theo, ta cần tìm thêm thông tin về doanh thu và thời lượng từ API TMDb, lưu vào file movies_metrics.csv

### **Lấy thông tin metrics từ TMDb API**

**Mục đích**: Lấy các thông số kinh tế quan trọng về mỗi phim không có trong file csv gốc.

**Dữ liệu lấy được** (từ API /movie/{tmdb_id}):
- **popularity**: Chỉ số nổi tiếng hiện tại (dựa trên lượt xem, tìm kiếm hôm nay)
- **vote_count**: Tổng số người đã đánh giá trên TMDb
- **vote_average**: Điểm trung bình từ TMDb (khác với MovieLens rating)
- **budget**: Vốn sản xuất (triệu USD)
- **revenue**: Doanh thu toàn cầu (triệu USD)
- **runtime**: Thời lượng phim (phút)

**Quy trình**:
1. Lặp qua tất cả ~9,700 phim trong df_links
2. Với mỗi phim, lấy tmdbId để gọi API
3. Lấy 7 thông số trên
4. Lưu vào DataFrame, thêm độ trễ giữa các request
5. Lưu kết quả vào `movies_metrics.csv` để không cần gọi API lại

**Lưu ý**: Dữ liệu từ API có thể thiếu (budget=0, revenue=0 nếu phim không có dữ liệu), sẽ xử lý ở bước sau.

In [255]:
def get_metrics_with_checkpoint(df, filename='movies_metrics_temp.csv'):
    """
    Hàm lấy metrics có khả năng lưu tạm (checkpoint) để không phải chạy lại từ đầu.
    """
    results = []
    
    # 1. Kiểm tra nếu đã có file chạy dở trước đó
    if os.path.exists(filename):
        df_checkpoint = pd.read_csv(filename)
        processed_ids = df_checkpoint['movieId'].tolist()
        results = df_checkpoint.to_dict('records')
        print(f"Đã tìm thấy checkpoint. Đang chạy tiếp từ phim thứ {len(processed_ids)}...")
    else:
        processed_ids = []

    # Lọc những phim chưa được xử lý
    df_todo = df[~df['movieId'].isin(processed_ids)].dropna(subset=['tmdbId'])
    
    print(f"Còn {len(df_todo)} phim cần lấy dữ liệu...")

    for i, (_, row) in enumerate(df_todo.iterrows(), 1):
        tmdb_id = int(row['tmdbId'])
        movie_id = row['movieId']
        url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?api_key={API_KEY}&language=en-US"
        
        # Thử lại tối đa 3 lần nếu lỗi timeout
        for attempt in range(3):
            # Sửa lại phần trong vòng lặp for attempt in range(3):
            try:
                response = requests.get(url, timeout=10)
                if response.status_code == 200:
                    data = response.json()
                    results.append({
                        'movieId': movie_id,
                        'tmdbId': tmdb_id,
                        'popularity': data.get('popularity'),
                        'vote_count': data.get('vote_count'),
                        'vote_average': data.get('vote_average'),
                        'budget': data.get('budget'),
                        'revenue': data.get('revenue'),
                        'runtime': data.get('runtime')
                    })
                    break # Thành công thì thoát vòng lặp attempt
                    
                elif response.status_code == 429: 
                    print(f"Rate limit exceeded (429), waiting...")
                    time.sleep(2)
                    
                else:
                    # THÊM ĐOẠN NÀY ĐỂ BIẾT NÓ ĐANG LỖI GÌ
                    print(f"Lỗi {response.status_code} tại movieId {movie_id}")
                    if response.status_code == 404: 
                        break # 404 là không tồn tại, không cần thử lại
                        
            except Exception as e:
                if attempt == 2: print(f"Bỏ qua movieId {movie_id} sau 3 lần thử thất bại. Lỗi: {e}")
                time.sleep(1)

        # Lưu checkpoint mỗi 50 phim
        if i % 50 == 0:
            pd.DataFrame(results).to_csv(filename, index=False)
            print(f"Đã lưu checkpoint tại phim thứ {len(results)}...")

    # Lưu kết quả cuối cùng
    df_final_metrics = pd.DataFrame(results)
    df_final_metrics.to_csv('movies_metrics.csv', index=False)
    return df_final_metrics

# Thực thi
df_metrics = get_metrics_with_checkpoint(df_links)

Đã tìm thấy checkpoint. Đang chạy tiếp từ phim thứ 9585...
Còn 153 phim cần lấy dữ liệu...
Lỗi 404 tại movieId 4207
Lỗi 404 tại movieId 4568
Lỗi 404 tại movieId 5069
Lỗi 404 tại movieId 5209
Lỗi 404 tại movieId 7646
Lỗi 404 tại movieId 7669
Lỗi 404 tại movieId 7762
Lỗi 404 tại movieId 7841
Lỗi 404 tại movieId 7842
Lỗi 404 tại movieId 26453
Lỗi 404 tại movieId 26587
Lỗi 404 tại movieId 26614
Lỗi 404 tại movieId 26649
Lỗi 404 tại movieId 26693
Lỗi 404 tại movieId 26761
Lỗi 404 tại movieId 26849
Lỗi 404 tại movieId 26887
Lỗi 404 tại movieId 27002
Lỗi 404 tại movieId 27036
Lỗi 404 tại movieId 27251
Lỗi 404 tại movieId 27611
Lỗi 404 tại movieId 27708
Lỗi 404 tại movieId 27751
Lỗi 404 tại movieId 31193
Lỗi 404 tại movieId 38198
Lỗi 404 tại movieId 49917
Lỗi 404 tại movieId 51024
Lỗi 404 tại movieId 52281
Lỗi 404 tại movieId 53883
Lỗi 404 tại movieId 55207
Lỗi 404 tại movieId 57772
Lỗi 404 tại movieId 61406
Lỗi 404 tại movieId 62336
Lỗi 404 tại movieId 62970
Lỗi 404 tại movieId 63433
Lỗi 404 

Những bộ film 404 là những bộ đã bị TMDb xóa khỏi database và nó chỉ chiếm khoảng 1.2%, tỉ lệ thấp nên bỏ không tìm kiếm nữa 


### **movies_metrics.csv**

**Mô tả**: Cung cấp các con số định lượng để xếp hạng mức độ "hot" của phim trên thị trường.

- popularity: Chỉ số độ nổi tiếng hiện tại (dựa trên lượt xem và tìm kiếm trong ngày).

- vote_count: Tổng số lượt người đã đánh giá bộ phim đó trên TMDb.

- budget: Số tiền (USD) đã bỏ ra để sản xuất phim.

- revenue: Tổng doanh thu (USD) mà bộ phim thu về được.

In [239]:
df_metrics = pd.read_csv('movies_metrics.csv')
inspect_data(df_metrics, "Movie Metrics Dataset")

 KIỂM TRA: MOVIE METRICS DATASET

1. THÔNG TIN TỔNG QUAN:
<class 'pandas.DataFrame'>
RangeIndex: 9623 entries, 0 to 9622
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   movieId       9623 non-null   int64  
 1   tmdbId        9623 non-null   int64  
 2   popularity    9623 non-null   float64
 3   vote_count    9623 non-null   int64  
 4   vote_average  9623 non-null   float64
 5   budget        9623 non-null   int64  
 6   revenue       9623 non-null   int64  
 7   runtime       9623 non-null   int64  
dtypes: float64(2), int64(6)
memory usage: 601.6 KB

2. DỮ LIỆU THIẾU (NULL):
-> Không có dữ liệu thiếu.

3. KIỂM TRA TRÙNG LẶP (DUPLICATES):
 - Số dòng trùng lặp hoàn toàn: 0

4. GIÁ TRỊ DUY NHẤT (UNIQUE):
 - Cột 'movieId': 9623 giá trị duy nhất
 - Cột 'tmdbId': 9623 giá trị duy nhất
 - Cột 'popularity': 8742 giá trị duy nhất
 - Cột 'vote_count': 3443 giá trị duy nhất
 - Cột 'vote_average': 2898 giá trị duy nhất


,movieId,tmdbId,popularity,vote_count,vote_average,budget,revenue,runtime
count,9623.000000,9.623000e+03,9623.000000,9623.000000,9623.000000,9.623000e+03,9.623000e+03,9623.000000
mean,41566.007794,5.412338e+04,2.804830,1696.350618,6.507452,1.898806e+07,5.676228e+07,104.358932
std,51817.930676,9.378989e+04,3.163183,3320.008729,0.860745,3.434845e+07,1.396903e+08,24.400925
min,1.000000,2.000000e+00,0.003700,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000
25%,3195.000000,9.622500e+03,1.040800,158.000000,6.000000,0.000000e+00,0.000000e+00,92.000000
50%,7176.000000,1.632500e+04,1.860700,494.000000,6.570000,4.400000e+06,7.985929e+06,102.000000
75%,74566.500000,4.340550e+04,3.483250,1628.000000,7.109000,2.300000e+07,4.739593e+07,115.000000
max,193609.000000,1.589361e+06,70.462700,38982.000000,8.900000,3.790000e+08,2.923706e+09,583.000000



6. DỮ LIỆU MẪU (HEAD):


,movieId,tmdbId,popularity,vote_count,vote_average,budget,revenue,runtime
0,1,862,21.1331,19587,7.970,30000000,401157969,81
1,2,8844,3.1936,11128,7.245,65000000,262821940,104
2,3,15602,1.5198,415,6.482,25000000,71500000,101
3,4,31357,1.6817,192,6.247,16000000,81452156,127
4,5,11862,2.0601,792,6.272,0,76594107,106


## **III. DATA INTEGRATION - Gộp dữ liệu từ 5 file CSV**

**Chiến lược gộp (Join Strategy)**:

Chúng ta sử dụng `pd.merge()` với `how='left'` để:**Giữ lại tất cả các dòng từ bảng "trái" (ratings) ngay cả khi không tìm thấy khớp trong bảng "phải"**

**Ba bước gộp**:

1. **Gộp Movies + Links** (thông tin cơ bản)
   - Bảng 1: movies.csv (title, genres)
   - Bảng 2: links.csv (imdbId, tmdbId từ API)
   - Khóa: movieId
   - Kết quả: Bảng có thông tin định danh cốt lõi

2. **Gộp kết quả với Metrics** (thông số kinh tế)
   - Thêm cột: popularity, vote_count, budget, revenue, runtime
   - Khóa: movieId
   - Kết quả: Bảng phim đầy đủ (df_movie_master)

3. **Gộp bảng Ratings lớn nhất với df_movie_master**
   - Mở rộng mỗi dòng rating bằng thông tin phim tương ứng
   - Khóa: movieId
   - Kết quả: Bảng tương tác (ratings + thông tin phim)

4. **Thêm Tags đã được gộp** (tùy chọn)
   - Một phim có thể có nhiều tags, nên gộp chúng thành một chuỗi
   - Sử dụng: `groupby()` + `apply(lambda)` để nối tất cả tags
   - Khóa: movieId
   - Kết quả: Bảng cuối cùng với tất cả thông tin

In [240]:
# BƯỚC 1: Tạo bảng thông tin phim đầy đủ (Full Movie Info)
# Gộp movies với links
df_movie_master = pd.merge(df_movies, df_links, on='movieId', how='left')

# Gộp tiếp với metrics (ví dụ: popularity, revenue)
# Lưu ý: Nếu metrics dùng tmdbId làm khóa, hãy đổi on='tmdbId'
df_movie_master = pd.merge(df_movie_master, df_metrics, on='movieId', how='left')

# BƯỚC 2: Gộp thông tin phim vào bảng Ratings (Bảng lớn nhất)
df_merge = pd.merge(df_ratings, df_movie_master, on='movieId', how='left')

# BƯỚC 3: Gộp thêm Tags (Tùy chọn)
# Vì một phim có thể có nhiều tags, việc merge này có thể làm tăng số dòng.
# Thường người ta sẽ gộp tags thành một chuỗi văn bản trước khi merge.
df_tags_combined = df_tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(set(str(v).lower() for v in x))).reset_index()
df_merge = pd.merge(df_merge, df_tags_combined, on='movieId', how='left')

print(f"Kích thước bảng cuối cùng sau khi merge 5 file: {df_merge.shape}")

Kích thước bảng cuối cùng sau khi merge 5 file: (100834, 16)


In [241]:
inspect_data(df_merge, "Final Merged Dataset")

 KIỂM TRA: FINAL MERGED DATASET

1. THÔNG TIN TỔNG QUAN:
<class 'pandas.DataFrame'>
RangeIndex: 100834 entries, 0 to 100833
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   userId        100834 non-null  int64  
 1   movieId       100834 non-null  int64  
 2   rating        100834 non-null  float64
 3   timestamp     100834 non-null  int64  
 4   title         100834 non-null  str    
 5   genres        100834 non-null  str    
 6   imdbId        100834 non-null  int64  
 7   tmdbId_x      100834 non-null  int64  
 8   tmdbId_y      100524 non-null  float64
 9   popularity    100524 non-null  float64
 10  vote_count    100524 non-null  float64
 11  vote_average  100524 non-null  float64
 12  budget        100524 non-null  float64
 13  revenue       100524 non-null  float64
 14  runtime       100524 non-null  float64
 15  tag           48287 non-null   str    
dtypes: float64(8), int64(5), str(3)
memory usage: 

,userId,movieId,rating,timestamp,imdbId,tmdbId_x,tmdbId_y,popularity,vote_count,vote_average,budget,revenue,runtime
count,100834.000000,100834.000000,100834.000000,1.008340e+05,1.008340e+05,1.008340e+05,1.005240e+05,100524.000000,100524.000000,100524.000000,1.005240e+05,1.005240e+05,100524.000000
mean,326.130849,19441.217456,3.501547,1.205940e+09,3.515505e+05,2.014379e+04,1.969325e+04,6.497197,6150.812841,7.031700,4.007519e+07,1.895450e+08,114.568680
std,182.618680,35543.516939,1.042537,2.162586e+08,6.220658e+05,5.377120e+04,5.276978e+04,5.963886,7050.575344,0.808717,4.714976e+07,2.613397e+08,24.046294
min,1.000000,1.000000,0.500000,8.281246e+08,4.170000e+02,2.000000e+00,2.000000e+00,0.003700,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000
25%,177.000000,1199.000000,3.000000,1.019124e+09,9.968500e+04,7.120000e+02,7.110000e+02,2.613200,1157.000000,6.511000,7.000000e+06,2.482262e+07,98.000000
50%,325.000000,2991.000000,3.500000,1.186087e+09,1.187710e+05,6.961000e+03,6.950000e+03,4.762900,3425.000000,7.085000,2.400000e+07,9.470000e+07,111.000000
75%,477.000000,8125.750000,4.000000,1.435994e+09,3.149790e+05,1.163500e+04,1.159500e+04,8.037900,8772.000000,7.600000,5.500000e+07,2.562713e+08,127.000000
max,610.000000,193609.000000,5.000000,1.537799e+09,8.391976e+06,1.589361e+06,1.589361e+06,70.462700,38982.000000,8.900000,3.790000e+08,2.923706e+09,583.000000



6. DỮ LIỆU MẪU (HEAD):


,userId,movieId,rating,timestamp,title,genres,imdbId,tmdbId_x,tmdbId_y,popularity,vote_count,vote_average,budget,revenue,runtime,tag
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,862.0,21.1331,19587.0,7.970,30000000.0,401157969.0,81.0,pixar fun
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,113228,15602,15602.0,1.5198,415.0,6.482,25000000.0,71500000.0,101.0,moldy old
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,113277,949,949.0,16.6753,8114.0,7.928,60000000.0,187400000.0,170.0,NaN
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,114369,807,807.0,19.0697,22681.0,8.379,33000000.0,327311859.0,127.0,twist ending serial killer mystery
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,114814,629,629.0,10.8187,11186.0,8.200,6000000.0,23300000.0,106.0,heist mindfuck tricky twist ending thriller su...


**Data review và data cleaning lần 2**

Có 5 vấn đề cần xử lý:

- "Cột sinh đôi": tmdbId_x và tmdbId_y

    - Tình trạng: tmdbId_x có ít Null hơn (chỉ 8), trong khi tmdbId_y thiếu tới 310.

    - Xử lý: Xóa một cột và giữ lại cột đầy đủ hơn.

- Cảnh báo "Duplicate"

    - Đây là hợp lệ. Vì đây là bảng gộp cuối cùng (Interaction), một phim được nhiều người xem nên movieId lặp lại là đúng. Một người xem nhiều phim nên userId lặp lại là đúng.

- Cột tag (Thiếu > 50%)

    - Cột này thiếu rất nhiều (52,547 dòng Null). Điều này dễ hiểu vì không phải ai xem phim cũng để lại tag.

    - Xử lý: Thay vì xóa dòng, hãy điền một giá trị rỗng hoặc "no_tag" để máy tính không bị lỗi khi xử lý văn bản.

- Xử lý dữ liệu số (Popularity, Budget, Revenue...) thiếu 310 dòng

    - Đây là những phim có trong bộ dữ liệu gốc nhưng không tìm thấy thông tin trên API/File phụ.

    - Xử lý: * Với runtime, popularity: Điền bằng giá trị trung vị (median). Với budget, revenue: Điền bằng 0.

### **Xử lý dữ liệu thiếu cột tag và số (Handling Missing Values in Multiple Columns)**

**Ba loại thiếu dữ liệu khác nhau, cách xử lý khác nhau:**

1. **Cột `tag` (thiếu > 50%, bình thường)**
   - Lý do: Không phải ai xem phim cũng để lại tag
   - Xử lý: Điền giá trị rỗng `''` để máy tính xử lý
   - Không xóa dòng (dữ liệu đánh giá vẫn quý giá)

2. **Cột `popularity`, `vote_count`, `runtime` (thiếu 318 dòng)**
   - Lý do: Phim những không tìm thấy trên TMDb
   - Xử lý: Điền bằng **giá trị trung vị (median)**
   - Tại sao median? Vì nó không bị ảnh hưởng bởi outlier (phim bom tấn)

3. **Cột `budget`, `revenue` (thiếu 318 dòng)**
   - Lý do: Dữ liệu xấp xỉ hoặc phim độc lập không công khai
   - Xử lý: Điền bằng **0** (tương đương "không có dữ liệu")
   - Lý do: Không nên dùng median vì 0 có ý nghĩa kinh tế (không có doanh số)

In [242]:
#drop cột tmdbId_y
df_merge.drop(columns=['tmdbId_y'], inplace=True)
df_merge.rename(columns={'tmdbId_x': 'tmdbId'}, inplace=True)

In [243]:
#xử lý giá trị thiếu trong cột tag
df_merge['tag'] = df_merge['tag'].fillna('')

In [244]:
#Xử lý dữ liệu số (Popularity, Budget, Revenue...)
cols_to_fix = ['popularity', 'vote_count', 'runtime', 'vote_average']
for col in cols_to_fix:
    df_merge[col] = df_merge[col].fillna(df_merge[col].median())

df_merge['budget'] = df_merge['budget'].fillna(0)
df_merge['revenue'] = df_merge['revenue'].fillna(0)

In [245]:
inspect_data(df_merge, "Final Merged Dataset")

 KIỂM TRA: FINAL MERGED DATASET

1. THÔNG TIN TỔNG QUAN:
<class 'pandas.DataFrame'>
RangeIndex: 100834 entries, 0 to 100833
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   userId        100834 non-null  int64  
 1   movieId       100834 non-null  int64  
 2   rating        100834 non-null  float64
 3   timestamp     100834 non-null  int64  
 4   title         100834 non-null  str    
 5   genres        100834 non-null  str    
 6   imdbId        100834 non-null  int64  
 7   tmdbId        100834 non-null  int64  
 8   popularity    100834 non-null  float64
 9   vote_count    100834 non-null  float64
 10  vote_average  100834 non-null  float64
 11  budget        100834 non-null  float64
 12  revenue       100834 non-null  float64
 13  runtime       100834 non-null  float64
 14  tag           100834 non-null  str    
dtypes: float64(7), int64(5), str(3)
memory usage: 11.5 MB

2. DỮ LIỆU THIẾU (NULL):
-> Không c

,userId,movieId,rating,timestamp,imdbId,tmdbId,popularity,vote_count,vote_average,budget,revenue,runtime
count,100834.000000,100834.000000,100834.000000,1.008340e+05,1.008340e+05,1.008340e+05,100834.000000,100834.000000,100834.000000,1.008340e+05,1.008340e+05,100834.000000
mean,326.130849,19441.217456,3.501547,1.205940e+09,3.515505e+05,2.014379e+04,6.491865,6142.432711,7.031864,3.995198e+07,1.889623e+08,114.557709
std,182.618680,35543.516939,1.042537,2.162586e+08,6.220658e+05,5.377120e+04,5.955485,7041.346144,0.807478,4.712948e+07,2.611486e+08,24.010115
min,1.000000,1.000000,0.500000,8.281246e+08,4.170000e+02,2.000000e+00,0.003700,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000
25%,177.000000,1199.000000,3.000000,1.019124e+09,9.968500e+04,7.120000e+02,2.616200,1158.000000,6.512000,6.800000e+06,2.450000e+07,98.000000
50%,325.000000,2991.000000,3.500000,1.186087e+09,1.187710e+05,6.961000e+03,4.762900,3425.000000,7.085000,2.400000e+07,9.372216e+07,111.000000
75%,477.000000,8125.750000,4.000000,1.435994e+09,3.149790e+05,1.163500e+04,8.035300,8703.000000,7.600000,5.500000e+07,2.550002e+08,127.000000
max,610.000000,193609.000000,5.000000,1.537799e+09,8.391976e+06,1.589361e+06,70.462700,38982.000000,8.900000,3.790000e+08,2.923706e+09,583.000000



6. DỮ LIỆU MẪU (HEAD):


,userId,movieId,rating,timestamp,title,genres,imdbId,tmdbId,popularity,vote_count,vote_average,budget,revenue,runtime,tag
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,21.1331,19587.0,7.970,30000000.0,401157969.0,81.0,pixar fun
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,113228,15602,1.5198,415.0,6.482,25000000.0,71500000.0,101.0,moldy old
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,113277,949,16.6753,8114.0,7.928,60000000.0,187400000.0,170.0,
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,114369,807,19.0697,22681.0,8.379,33000000.0,327311859.0,127.0,twist ending serial killer mystery
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,114814,629,10.8187,11186.0,8.200,6000000.0,23300000.0,106.0,heist mindfuck tricky twist ending thriller su...


## **IV. DATA REDUCTION**

Giảm số cột (Feature Selection): những cột không cần thiết: tmdbId, imdbId, timestamp

### **Lựa chọn các cột quan trọng (Feature Selection)**

**Mục đích**: Giảm bớt số cột không cần thiết để:
- Giống từ mô hình (model) dễ dàng 
- Giảm dung lượng bộ nhớ
- Tập trung vào đặc trưng quan trọng

**Các cột được giữ lại (11 cột)**:
- **userId**: Mã người dùng (để xác định ai đánh giá)
- **movieId**: Mã phim (khóa chính)
- **rating**: Điểm đánh giá người dùng (0.5-5.0)
- **title**: Tên phim (để xác định)
- **genres**: Thể loại (đặc trưng nội dung)
- **popularity**: Độ nổi tiếng hiện tại (lấy từ TMDb API)
- **vote_count**: Số lượt đánh giá trên TMDb
- **budget**: Vốn sản xuất (triệu USD)
- **revenue**: Doanh thu (triệu USD)
- **runtime**: Thời lượng (phút)
- **tag**: Từ khóa của người dùng
- **vote_average**:

**Các cột bị xóa**:
- `tmdbId`, `imdbId`: ID ngoài (không cần cho phân tích)
- `timestamp`: Thời gian đánh giá (ít quan trọng cho khuyến nghị phim)

In [246]:
# Danh sách các cột quan trọng muốn giữ lại
important_features = [
    'userId', 'movieId', 'rating', 'title', 'genres', 
    'popularity', 'vote_count', 'budget', 'revenue', 'runtime', 'tag', 'vote_average'
]

# Tạo DataFrame mới chỉ chứa các cột này
df_reduced = df_merge[important_features].copy()

print(f"{df_reduced.head()}")
print('\n')

   userId  movieId  rating                        title  \
0       1        1     4.0             Toy Story (1995)   
1       1        3     4.0      Grumpier Old Men (1995)   
2       1        6     4.0                  Heat (1995)   
3       1       47     5.0  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0   Usual Suspects, The (1995)   

                                        genres  popularity  vote_count  \
0  Adventure|Animation|Children|Comedy|Fantasy     21.1331     19587.0   
1                               Comedy|Romance      1.5198       415.0   
2                        Action|Crime|Thriller     16.6753      8114.0   
3                             Mystery|Thriller     19.0697     22681.0   
4                       Crime|Mystery|Thriller     10.8187     11186.0   

       budget      revenue  runtime  \
0  30000000.0  401157969.0     81.0   
1  25000000.0   71500000.0    101.0   
2  60000000.0  187400000.0    170.0   
3  33000000.0  327311859.0    127.0   
4   6

## **V. DATA TRANSFORMATION**

**Mục đích**: Biến đổi và chuẩn hóa dữ liệu về định dạng, phạm vi giá trị, và mã hóa văn bản để chuẩn bị cho mô hình Machine Learning.

### **Chuẩn hóa định dạng dữ liệu (Format Standardization)**

**Các bước thực hiện**:

1. **Chuyển Genres từ chuỗi thành List**
   - Dữ liệu gốc: `"Action|Adventure|Sci-Fi"` (chuỗi)
   - Sử dụng: `.str.split('|')` để tách từng thể loại
   - Dữ liệu sau: `['Action', 'Adventure', 'Sci-Fi']` (danh sách)
   - **Lợi ích**: Dễ xử lý từng thể loại riêng và kiểm tra các phim thuộc loại nào

2. **Tính điểm trung bình (avg_rating) cho mỗi phim**
   - Nhóm (groupby) tất cả các đánh giá theo `movieId`
   - Tính trung bình cộng của các rating từ tất cả người dùng
   - Thêm cột mới `avg_rating` vào bảng
   - **Lợi ích**: Đánh giá mức độ nổi tiếng của mỗi phim từ cộng đồng

3. **Gộp cột mới vào bảng chính**
   - Sử dụng `merge()` để kết nối bảng với cột avg_rating

In [247]:
df_normalized = df_reduced.copy()

# 1. Biến genres từ chuỗi thành List
# Chúng ta dùng hàm split('|') để tách chuỗi tại ký tự gạch đứng
df_normalized['genres'] = df_normalized['genres'].fillna('').str.split('|')

# 2. Tính điểm trung bình (avg_rating) cho mỗi phim
# Tính từ bảng gốc hoặc bảng hiện tại (miễn là có đủ movieId và rating)
movie_avg = df_normalized.groupby('movieId')['rating'].mean().reset_index()
movie_avg.rename(columns={'rating': 'avg_rating'}, inplace=True)

# 3. Merge cột avg_rating vào dataframe chính
df_normalized = pd.merge(df_normalized, movie_avg, on='movieId', how='left')

# Hiển thị kết quả kiểm tra
print("Dữ liệu sau khi biến đổi:")
print(df_normalized[['title', 'genres', 'avg_rating']].head())

Dữ liệu sau khi biến đổi:
                         title  \
0             Toy Story (1995)   
1      Grumpier Old Men (1995)   
2                  Heat (1995)   
3  Seven (a.k.a. Se7en) (1995)   
4   Usual Suspects, The (1995)   

                                              genres  avg_rating  
0  [Adventure, Animation, Children, Comedy, Fantasy]    3.920930  
1                                  [Comedy, Romance]    3.259615  
2                          [Action, Crime, Thriller]    3.946078  
3                                [Mystery, Thriller]    3.975369  
4                         [Crime, Mystery, Thriller]    4.237745  


### **Chuẩn hóa dữ liệu số (Normalization)**

**Phương pháp**: Min-Max Scaling - Dịch chuyển và co giãn dữ liệu để tất cả các cột nằm trong khoảng [0, 1].

**Công thức**: $ \text{X\_scaled} = \frac{X - X_{min}}{X_{max} - X_{min}} $

**Tại sao cần chuẩn hóa?**
- **Các cột có thang đo khác nhau**: Ví dụ, 'budget' tính bằng triệu USD (0-300 triệu) còn 'runtime' tính bằng phút (80-240). Nếu không chuẩn hóa, thuật toán ML sẽ bị ảnh hưởng bởi thang đo lớn.
- **Cải thiện hiệu suất mô hình**: Nhiều thuật toán (KNN, Neural Networks, SVM) hoạt động tốt hơn khi dữ liệu được chuẩn hóa.
- **Đảm bảo công bằng khi so sánh**: Tất cả các đặc trưng đều có ảnh hưởng tương đương.

**Các cột được chuẩn hóa**: popularity, vote_count, budget, revenue, runtime, avg_rating

In [248]:
# Danh sách các cột số cần chuẩn hóa
cols_to_scale = ['popularity', 'vote_count', 'budget', 'revenue', 'runtime', 'avg_rating']

# BƯỚC 1: Kiểm tra và xử lý giá trị âm (nếu có)
print("=== KIỂM CHỨNG DỮ LIỆU TRƯỚC LOG TRANSFORM ===")
for col in cols_to_scale:
    min_val = df_normalized[col].min()
    max_val = df_normalized[col].max()
    neg_count = (df_normalized[col] < 0).sum()
    print(f"{col}: Min={min_val:.2f}, Max={max_val:.2f}, Giá trị âm={neg_count}")

# BƯỚC 2: Log Transform để giảm skewness
df_normalized_log = df_normalized.copy()
for col in cols_to_scale:
    # Xử lý giá trị âm (nếu có) - Set thành 0 trước khi log
    if df_normalized_log[col].min() < 0:
        df_normalized_log[col] = df_normalized_log[col].clip(lower=0)
        print(f"⚠️  {col}: Đã xử lý {(df_normalized_log[col] < 0).sum()} giá trị âm")
    
    # Áp dụng log(1+x) để tránh log(0)
    if df_normalized_log[col].min() >= 0:
        df_normalized_log[col] = np.log1p(df_normalized_log[col])

print("\n✓ Đã áp dụng Log Transform")

# BƯỚC 3: Khởi tạo scaler và chuẩn hóa MinMax
scaler = MinMaxScaler()
df_normalized[cols_to_scale] = scaler.fit_transform(df_normalized_log[cols_to_scale])

print("✓ Đã áp dụng Min-Max Scaling sau Log Transform")

# Kiểm tra lại 5 dòng đầu
print("\nDữ liệu sau khi chuẩn hóa (Log + Min-Max):")
display(df_normalized.head())


=== KIỂM CHỨNG DỮ LIỆU TRƯỚC LOG TRANSFORM ===
popularity: Min=0.00, Max=70.46, Giá trị âm=0
vote_count: Min=0.00, Max=38982.00, Giá trị âm=0
budget: Min=0.00, Max=379000000.00, Giá trị âm=0
revenue: Min=0.00, Max=2923706026.00, Giá trị âm=0
runtime: Min=0.00, Max=583.00, Giá trị âm=0
avg_rating: Min=0.50, Max=5.00, Giá trị âm=0

✓ Đã áp dụng Log Transform
✓ Đã áp dụng Min-Max Scaling sau Log Transform

Dữ liệu sau khi chuẩn hóa (Log + Min-Max):


,userId,movieId,rating,title,genres,popularity,vote_count,budget,revenue,runtime,tag,vote_average,avg_rating
0,1,1,4.0,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",0.725212,0.934896,0.871598,0.908871,0.691803,pixar fun,7.970,0.856984
1,1,3,4.0,Grumpier Old Men (1995),"[Comedy, Romance]",0.215799,0.570500,0.862368,0.829744,0.726067,moldy old,6.482,0.752880
2,1,6,4.0,Heat (1995),"[Action, Crime, Thriller]",0.672485,0.851534,0.906688,0.873952,0.807181,,7.928,0.860661
3,1,47,5.0,Seven (a.k.a. Se7en) (1995),"[Mystery, Thriller]",0.702269,0.948769,0.876423,0.899537,0.761712,twist ending serial killer mystery,8.379,0.864921
4,1,50,5.0,"Usual Suspects, The (1995)","[Crime, Mystery, Thriller]",0.578127,0.881905,0.790120,0.778302,0.733580,heist mindfuck tricky twist ending thriller su...,8.200,0.901992


### **Feature Engineering từ Text - Chuyển đổi Text thành Số**

**Vấn đề:** Mô hình Machine Learning chỉ hiểu **số**, nhưng dữ liệu của chúng ta có:
- **genres**: `['Action', 'Adventure']` (danh sách text)
- **tags**: `'action inspiring thrilling'` (chuỗi text)

**Giải pháp**: Chuyển text thành vector/ma trận số để mô hình có thể học.

#### **Chiến lược 1: OneHot Encoding cho Genres**

**Khái niệm**: Mỗi thể loại là 1 cột (0 hoặc 1)

Ví dụ: `['Action', 'Adventure']` → `[1, 1, 0, 0, 0, 0...]`

**Lợi ích**: 
- Dễ tính độ tương tự giữa 2 phim (dùng cosine similarity)
- Dễ xây dựng content-based recommendation

#### **Chiến lược 2: TF-IDF cho Tags**

**Khái niệm**: Tính trọng số từ khóa (từ quan trọng với phim cụ thể = trọng số cao)

- **TF** (Term Frequency): Tần suất từ trong phim
- **IDF** (Inverse Document Frequency): Độ hiếm của từ trong toàn bộ dataset
- **TF-IDF** = TF × IDF

**Lợi ích**:
- ✓ Chuyển text → số (mô hình có thể xử lý)
- ✓ Tạo content vectors (để tính similarity)
- ✓ Highlight từ khóa quan trọng (TF-IDF)
- ✓ Sẵn sàng cho Collaborative Filtering & Content-Based Filtering

In [249]:
# Kiểm tra và làm sạch genres trước khi OneHot
print("=== KIỂM CHỨNG GENRES TRƯỚC ENCODING ===")
print(f"Dòng có genres null: {df_normalized['genres'].isna().sum()}")
print(f"Dòng có genres rỗng: {(df_normalized['genres'] == '').sum()}")
print(f"Dòng có genres list: {sum(isinstance(x, list) for x in df_normalized['genres'])}")

# Bước 1: Lấy tất cả thể loại duy nhất
all_genres = set()
for genres_list in df_normalized['genres'].dropna():
    if isinstance(genres_list, list):
        # Lọc bỏ các phần tử rỗng hoặc whitespace
        clean_genres = [g.strip() for g in genres_list if g and str(g).strip()]
        all_genres.update(clean_genres)

all_genres = sorted(list(all_genres))
print(f"\n✓ Số thể loại duy nhất: {len(all_genres)}")
print(f"Danh sách: {all_genres[:10]}...") # Hiển thị 10 thể loại đầu

# Bước 2: Tạo OneHot encoding với xác thực
genre_vectors = []
unmapped_genres = set()  # Theo dõi các thể loại không xác định

for genres_list in df_normalized['genres']:
    vector = [0] * len(all_genres)
    if isinstance(genres_list, list):
        for g in genres_list:
            g_clean = str(g).strip() if g else ''
            if g_clean in all_genres:
                idx = all_genres.index(g_clean)
                vector[idx] = 1
            elif g_clean:  # Chỉ cảnh báo nếu không rỗng
                unmapped_genres.add(g_clean)
    genre_vectors.append(vector)

if unmapped_genres:
    print(f"⚠️  Cảnh báo: {len(unmapped_genres)} thể loại không được mapping")

# Bước 3: Chuyển thành DataFrame
genre_encoded = pd.DataFrame(
    genre_vectors, 
    columns=[f'genre_{g}' for g in all_genres],
    index=df_normalized.index
)

print(f"\n✓ Genre Encoded Shape: {genre_encoded.shape}")
print("Mẫu OneHot Encoding:")
display(genre_encoded.head())

# Bước 4: Gộp vào dataframe gốc (reset index để tránh mismatch)
df_normalized = pd.concat([df_normalized.reset_index(drop=True), genre_encoded.reset_index(drop=True)], axis=1)
print(f"✓ Kích thước df_normalized sau OneHot: {df_normalized.shape}")


=== KIỂM CHỨNG GENRES TRƯỚC ENCODING ===
Dòng có genres null: 0
Dòng có genres rỗng: 0
Dòng có genres list: 100834

✓ Số thể loại duy nhất: 20
Danh sách: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy']...

✓ Genre Encoded Shape: (100834, 20)
Mẫu OneHot Encoding:


,genre_(no genres listed),genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,genre_Fantasy,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0
4,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0


✓ Kích thước df_normalized sau OneHot: (100834, 33)


In [250]:
# Bước 1: Chuẩn bị dữ liệu tag - làm sạch và xác thực
df_normalized['tag'] = df_normalized['tag'].fillna('').str.strip().str.lower()

# Xóa các tag chỉ chứa khoảng trắng
df_normalized['tag'] = df_normalized['tag'].apply(
    lambda x: x if x and str(x).strip() else ''
)

print("=== KIỂM CHỨNG DATA TAGS ===")
print(f"Phim có tag (không rỗng): {(df_normalized['tag'] != '').sum()}")
print(f"Phim không có tag: {(df_normalized['tag'] == '').sum()}\n")

# Bước 2: Kiểm tra xem có đủ tag không (nếu quá ít thì bỏ qua TF-IDF)
tag_count = (df_normalized['tag'] != '').sum()
if tag_count < 10:
    print("⚠️  CẢNH BÁO: Quá ít tag để tính TF-IDF. Bỏ qua bước này.")
    tag_features = pd.DataFrame(index=df_normalized.index)
else:
    # Bước 3: Khởi tạo TfidfVectorizer
    tfidf = TfidfVectorizer(
        max_features=100,      # Giữ top 100 từ quan trọng nhất
        min_df=2,              # Từ phải xuất hiện trong ít nhất 2 phim
        max_df=0.7,            # Từ không được xuất hiện quá 70% phim
        ngram_range=(1, 2),    # Cho phép từ đơn và 2-gram
        stop_words='english'   # Loại bỏ từ phổ biến
    )

    # Bước 4: Fit và transform
    tag_tfidf = tfidf.fit_transform(df_normalized['tag'])

    print(f"TF-IDF Matrix Shape: {tag_tfidf.shape}")
    print(f"Số từ được giữ lại: {len(tfidf.get_feature_names_out())}\n")

    # Bước 5: Chuyển thành DataFrame
    tag_features = pd.DataFrame(
        tag_tfidf.toarray(),
        columns=[f'tag_tfidf_{name}' for name in tfidf.get_feature_names_out()],
        index=df_normalized.index
    )

    print("Mẫu TF-IDF Encoding:")
    display(tag_features.head())

# Bước 6: Gộp vào dataframe gốc (reset index để tránh mismatch)
df_normalized = pd.concat([df_normalized.reset_index(drop=True), tag_features.reset_index(drop=True)], axis=1)
print(f"\n✓ Kích thước df_normalized sau TF-IDF: {df_normalized.shape}")


=== KIỂM CHỨNG DATA TAGS ===
Phim có tag (không rỗng): 48287
Phim không có tag: 52547

TF-IDF Matrix Shape: (100834, 100)
Số từ được giữ lại: 100

Mẫu TF-IDF Encoding:


,tag_tfidf_250,tag_tfidf_action,tag_tfidf_adventure,tag_tfidf_aliens,tag_tfidf_apocalyptic,tag_tfidf_appealing,tag_tfidf_atmospheric,tag_tfidf_bad,tag_tfidf_based,tag_tfidf_best,...,tag_tfidf_time travel,tag_tfidf_timeline,tag_tfidf_travel,tag_tfidf_twist,tag_tfidf_twist ending,tag_tfidf_violence,tag_tfidf_violent,tag_tfidf_visually,tag_tfidf_visually appealing,tag_tfidf_war
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.580507,0.585944,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.357901,0.361253,0.0,0.0,0.0,0.0,0.0



✓ Kích thước df_normalized sau TF-IDF: (100834, 133)


## **VI. DATA DISCRETIZATION**

### **Rời rạc hóa dữ liệu (Binning/Categorization)**

**Mục đích**: Chuyển đổi các giá trị liên tục (continuous) thành các danh mục rõ ràng (categorical) để:
- Giảm độ phức tạp của dữ liệu
- Tạo ra các "nhóm" có ý nghĩa kinh doanh (ví dụ: Phim "cũ" vs "mới", Phim "ngắn" vs "dài")
- Chuẩn bị dữ liệu cho một số thuật toán phân loại

**Hai chiến lược:**
1. **Binning đều (Fixed Bins)**: Chia theo các mốc cố định (ví dụ: năm 1970, 1990, 2005...)
   - Dễ hiểu, phù hợp với các quy tắc kinh doanh
2. **Binning tự động (Quantile-based)**: Chia sao cho các nhóm có số lượng mẫu tương đương
   - Phân bố dữ liệu đều, tránh nhóm quá nhỏ

### **Trích xuất năm và phân loại thời đại (Year Extraction & Era Binning)**

**Bước 1: Trích xuất năm từ cột title**
- Dữ liệu gốc: `"Toy Story (1995)"`
- Sử dụng Regex `\((\d{4})\)` để tìm 4 chữ số trong ngoặc
- Kết quả: Cột mới `year` = 1995
- Mặc định: Nếu không tìm thấy năm, đặt = 2000

**Bước 2: Rời rạc hóa năm thành các "Era" (Thời đại)**
- Sử dụng `pd.cut()` để chia khoảng năm thành 5 nhóm:
  - **Classic** (≤1970): Phim kinh điển, sơ khai (ít có dữ liệu về khán giả hiện đại)
  - **Retro** (1970-1990): Phim cổ điển thế hệ 80s
  - **Modern_Transition** (1990-2005): Chuyển tiếp analog → số hóa
  - **Digital_Age** (2005-2015): Thế hệ CGI & streaming bắt đầu
  - **Current** (2015-2026): Phim hiện đại, streaming mainstream

**Lợi ích**:
- Phát hiện mối quan hệ giữa thời đại công chiếu và rating/doanh thu
- Xác định mưu xu phim qua các thập kỷ
- Đơn giản hơn so với 100+ giá trị năm khác nhau

In [251]:
df_discretized = df_normalized.copy()

# 1. Trích xuất năm từ title bằng Regex
def extract_year(title):
    year = re.search(r'\((\d{4})\)', title)
    return int(year.group(1)) if year else 2000 # Mặc định 2000 nếu ko tìm thấy

df_discretized['year'] = df_discretized['title'].apply(extract_year)

# Kiểm chứng năm
print("=== KIỂM CHỨNG DỮ LIỆU NĂM ===")
print(f"Năm min: {df_discretized['year'].min()}")
print(f"Năm max: {df_discretized['year'].max()}")
print(f"Không tìm thấy năm (year=2000): {(df_discretized['year'] == 2000).sum()}")

# 2. Rời rạc hóa Năm thành các Era (Thời đại)
# Mở rộng khoảng để bao gồm tất cả năm (kể cả năm 0)
bins = [0, 1970, 1990, 2005, 2015, 2026]
labels = ['Classic', 'Retro', 'Modern_Transition', 'Digital_Age', 'Current']
df_discretized['movie_era'] = pd.cut(df_discretized['year'], bins=bins, labels=labels, right=False)

# Kiểm tra xem có giá trị null sau cut không
if df_discretized['movie_era'].isna().any():
    print(f"⚠️  Cảnh báo: {df_discretized['movie_era'].isna().sum()} giá trị year không được map vào era")
    print(f"Các năm bị lỗi: {df_discretized[df_discretized['movie_era'].isna()]['year'].unique()}")

print(f"\nPhân bố Era:\n{df_discretized['movie_era'].value_counts().sort_index()}")

display(df_discretized[['title', 'year', 'movie_era']].head(10))


=== KIỂM CHỨNG DỮ LIỆU NĂM ===
Năm min: 1902
Năm max: 2018
Không tìm thấy năm (year=2000): 4284

Phân bố Era:
movie_era
Classic               6571
Retro                17907
Modern_Transition    55462
Digital_Age          18469
Current               2425
Name: count, dtype: int64


,title,year,movie_era
0,Toy Story (1995),1995,Modern_Transition
1,Grumpier Old Men (1995),1995,Modern_Transition
2,Heat (1995),1995,Modern_Transition
3,Seven (a.k.a. Se7en) (1995),1995,Modern_Transition
4,"Usual Suspects, The (1995)",1995,Modern_Transition
5,From Dusk Till Dawn (1996),1996,Modern_Transition
6,Bottle Rocket (1996),1996,Modern_Transition
7,Braveheart (1995),1995,Modern_Transition
8,Rob Roy (1995),1995,Modern_Transition
9,Canadian Bacon (1995),1995,Modern_Transition


### **Chia Runtime thành các danh mục (Runtime Binning)**

**Phương pháp**: `pd.qcut()` - Chia dữ liệu thành 3 nhóm có số lượng mẫu bằng nhau (Quantile-based).

**Kết quả**:
- **Short**: Phim ngắn/Hoạt hình (dưới ~95 phút) - thường có kĩ xảo hơn
- **Medium**: Phim chuẩn (~95-115 phút) - phổ biến nhất
- **Long**: Phim dài/Bom tấn (trên ~115 phút) - thường có budget cao hơn

**Lợi ích**:
- Phát hiện mối quan hệ giữa thời lượng và các yếu tố khác (rating, doanh thu, tiêu điểm khán giả)
- Tạo ra các "profile" phim thuận tiện cho phân tích và gợi ý

In [252]:
# Chia Runtime thành 3 nhóm dựa trên phân phối dữ liệu
# Short: Phim ngắn/Hoạt hình, Medium: Phim chuẩn, Long: Phim dài/Bom tấn
df_discretized['runtime_category'] = pd.qcut(df_discretized['runtime'], q=3, labels=['Short', 'Medium', 'Long'])
display(df_discretized[['title', 'runtime', 'runtime_category']].head(10))


,title,runtime,runtime_category
0,Toy Story (1995),0.691803,Short
1,Grumpier Old Men (1995),0.726067,Short
2,Heat (1995),0.807181,Long
3,Seven (a.k.a. Se7en) (1995),0.761712,Long
4,"Usual Suspects, The (1995)",0.733580,Medium
5,From Dusk Till Dawn (1996),0.736487,Medium
6,Bottle Rocket (1996),0.709868,Short
7,Braveheart (1995),0.814359,Long
8,Rob Roy (1995),0.775780,Long
9,Canadian Bacon (1995),0.709868,Short


---

## **VII. SPARSITY ANALYSIS - Phân tích độ Thưa của Dữ liệu**

### **Khái niệm Sparsity**

**Sparsity** (độ thưa) = Tỷ lệ giá trị thiếu / tổng số giá trị trong rating matrix.

**Rating Matrix** - Cấu trúc dữ liệu:
```
        Movie 1  Movie 2  Movie 3  ...  Movie N
User 1    4.5      NaN     3.2    ...    NaN
User 2    NaN      5.0    4.0     ...    3.8
User 3    2.0     3.5     NaN     ...    4.2
...
User M    NaN      NaN    5.0     ...    NaN
```

**Tại sao phải phân tích?**
- **Sparsity cao** (>95%) = Rất ít dữ liệu → Khó xây dựng mô hình → Cần Regularization hoặc Matrix Factorization
- **Sparsity thấp** (<70%) = Dữ liệu dầy đủ → Có thể dùng các phương pháp đơn giản hơn
- Giúp quyết định chọn thuật toán phù hợp:
  - Sparsity cao → Dùng SVD, NMF, Embedding
  - Sparsity trung bình → Dùng KNN hoặc Neural Networks
  - Sparsity thấp → Có thể dùng Linear Regression

In [253]:
# Tạo rating matrix: hàng = user, cột = movie, giá trị = rating
rating_matrix = df_discretized.pivot_table(index='userId', columns='movieId', values='rating')

print("=== RATING MATRIX STRUCTURE ===")
print(f"Kích thước: {rating_matrix.shape[0]} users × {rating_matrix.shape[1]} movies")
print(f"Tổng giá trị có thể: {rating_matrix.shape[0] * rating_matrix.shape[1]:,}")

# Tính Sparsity
total_cells = rating_matrix.shape[0] * rating_matrix.shape[1]
filled_cells = rating_matrix.notna().sum().sum()
empty_cells = total_cells - filled_cells

sparsity = (empty_cells / total_cells) * 100
density = (filled_cells / total_cells) * 100

print(f"\n=== SPARSITY ANALYSIS ===")
print(f"Ô có dữ liệu: {filled_cells:,} ({density:.2f}%)")
print(f"Ô thiếu dữ liệu: {empty_cells:,} ({sparsity:.2f}%)")

# Phân tích theo user
user_ratings = df_discretized.groupby('userId').size()
print(f"\n=== USER ANALYSIS ===")
print(f"Trung bình mỗi user đánh giá: {user_ratings.mean():.1f} phim")
print(f"User hoạt động nhất: {user_ratings.max()} phim")
print(f"User ít hoạt động nhất: {user_ratings.min()} phim")

# Phân tích theo movie
movie_ratings = df_discretized.groupby('movieId').size()
print(f"\n=== MOVIE ANALYSIS ===")
print(f"Trung bình mỗi phim được đánh giá: {movie_ratings.mean():.1f} lần")
print(f"Phim được yêu thích nhất: {movie_ratings.max()} đánh giá")
print(f"Phim ít được xem nhất: {movie_ratings.min()} đánh giá")

# Phân bố số rating trên user
print(f"\n=== RATING DISTRIBUTION ===")
print(user_ratings.describe())

=== RATING MATRIX STRUCTURE ===
Kích thước: 610 users × 9720 movies
Tổng giá trị có thể: 5,929,200

=== SPARSITY ANALYSIS ===
Ô có dữ liệu: 100,834 (1.70%)
Ô thiếu dữ liệu: 5,828,366 (98.30%)

=== USER ANALYSIS ===
Trung bình mỗi user đánh giá: 165.3 phim
User hoạt động nhất: 2698 phim
User ít hoạt động nhất: 20 phim

=== MOVIE ANALYSIS ===
Trung bình mỗi phim được đánh giá: 10.4 lần
Phim được yêu thích nhất: 329 đánh giá
Phim ít được xem nhất: 1 đánh giá

=== RATING DISTRIBUTION ===
count     610.000000
mean      165.301639
std       269.477828
min        20.000000
25%        35.000000
50%        70.500000
75%       168.000000
max      2698.000000
dtype: float64


In [254]:
# --- 1. Định nghĩa các nhóm cột ---
metadata_cols = ['userId', 'movieId', 'title', 'tag', 'genres', 'movie_era', 'runtime_category']

# Logic để lấy các nhóm cột đặc trưng
genre_cols = [c for c in df_discretized.columns if c.startswith('genre_')]
tfidf_cols = [c for c in df_discretized.columns if c.startswith('tag_tfidf_')]
numeric_cols = ['popularity', 'vote_count', 'budget', 'revenue', 'runtime', 
                'vote_average', 'avg_rating', 'year']

feature_cols = genre_cols + tfidf_cols + numeric_cols

# --- 2. Tạo các tập dữ liệu con ---
# File Metadata: Dùng để hiển thị tên phim, tra cứu thông tin
df_metadata = df_discretized[metadata_cols]

# File Features: Chỉ chứa các giá trị số để train mô hình
df_features = df_discretized[feature_cols]

# --- 3. Lưu thành các file CSV ---
df_metadata.to_csv('movies_metadata.csv', index=False)
df_features.to_csv('model_features.csv', index=False)

print("✓ Đã xuất 2 file thành công:")
print("- movies_metadata.csv (Tra cứu)")
print("- model_features.csv (Đầu vào mô hình)")

✓ Đã xuất 2 file thành công:
- movies_metadata.csv (Tra cứu)
- model_features.csv (Đầu vào mô hình)


## **VII. BÁO CÁO QUỸ TRÌNH XỬ LÍ DỮ LIỆU**

### **BAN ĐẦU CÓ:**

| File | Dòng | Mô Tả |
|------|------|-------|
| **movies.csv** | 9,742 | ID, tiêu đề, thể loại phim |
| **ratings.csv** | 100,836 | User ID, Movie ID, đánh giá, thời gian |
| **tags.csv** | 3,683 | User ID, Movie ID, tag, timestamp |
| **links.csv** | 9,742 | Movie ID, IMDb ID, TMDb ID |
| **movies_metadata.csv** | 45,463 | Thông tin phim từ TMDb (budget, revenue, runtime...) |

**Tổng cộng: ~169,466 dòng dữ liệu từ 5 nguồn khác nhau**

---

### **QUÁ TRÌNH XỬ LÍ (6 BƯỚC CHÍNH):**

| Bước | Tên | Công Việc Chính | Thuật Toán / Kỹ Thuật |
|------|-----|-----------------|----------------------|
| **1** | Review Dataset | Tải và kiểm tra cấu trúc 5 file CSV | Pandas (read_csv, info, describe) |
| **2** | Data Cleaning | Xóa dòng null, trùng lặp, giá trị bất hợp lệ | Pandas dropna(), drop_duplicates(), filtering logic |
| **3** | Data Integration | Ghép 5 bảng thành 1 bảng tổng hợp (join movies + ratings + tags + links + metadata) | SQL JOIN (inner/left join, merge) |
| **4** | Data Reduction | Chọn cột quan trọng, loại bỏ cột không cần | Feature selection, Column selection |
| **5** | Data Transformation | Chuẩn hóa dữ liệu số, tính toán đặc trưng mới (avg_rating, tfidf), phân nhóm năm | MinMaxScaler, TF-IDF, Aggregation, Binning |
| **6** | Data Discretization | Rời rạc hóa dữ liệu liên tục thành danh mục (runtime_category, movie_era) | Binning, Cut/Qcut, Bucketing, One-Hot Encoding |
---

### **KẾT QUẢ CÓ ĐƯỢC:**

| Output | Dòng | Cột | Mục Đích|
|--------|------|-----|---------|
| **movies_metadata.csv** | - | 7 | Tra cứu thông tin phim (userId, movieId, title, tag, genres, movie_era, runtime_category) |
| **model_features.csv** | - | ~110+ | Đầu vào cho mô hình ML (các cột genre, tfidf, và dữ liệu số) |
| **ratings.csv** | 100,836 | 5 | Dữ liệu đánh giá của user (userId, movieId, rating, timestamp, avg_rating) |


**Các đặc trưng được tạo:**
- ✓ **Genre Columns**: Các cột nhị phân cho từng thể loại phim (genre_Action, genre_Comedy...) → **One-Hot Encoding**
- ✓ **TF-IDF Features**: Vector từ khóa được trích xuất từ tags (tag_tfidf_0 đến tag_tfidf_N) → **TF-IDF Vectorizer**
- ✓ **Normalized Features**: Dữ liệu số chuẩn hóa (popularity, score, budget, revenue, runtime, rating...) → **MinMaxScaler**
- ✓ **New Features**: Rating trung bình, phân loại runtime, giai đoạn phim (movie_era) → **Aggregation, Binning/Cut**

In [ ]:
# Lưu tất cả các bảng CSV sau khi xử lý với hậu tố *_cleaning
def _safe_save_df(df_name: str, filename: str):
    if df_name in globals():
        globals()[df_name].to_csv(filename, index=False)
        print(f"- Đã lưu {filename} từ biến {df_name}")
    else:
        print(f"- Bỏ qua {filename} (không tìm thấy biến {df_name})")

print("✓ Đang lưu các file CSV phiên bản _cleaning ...")

# 1. Các bảng gốc sau khi đã được làm sạch / cập nhật
_safe_save_df('df_links', 'links_cleaning.csv')
_safe_save_df('df_movies', 'movies_cleaning.csv')
_safe_save_df('df_ratings', 'ratings_cleaning.csv')
_safe_save_df('df_tags', 'tags_cleaning.csv')
_safe_save_df('df_metrics', 'movies_metrics_cleaning.csv')

# 2. Các bảng sau khi tích hợp & biến đổi đặc trưng
_safe_save_df('df_discretized', 'movies_dataset_cleaning.csv')
_safe_save_df('df_metadata', 'movies_metadata_cleaning.csv')
_safe_save_df('df_features', 'model_features_cleaning.csv')


: 